#

In [1]:
import pandas as pd

In [8]:
ip_port = "132.72.65.48:9996"

In [9]:
url = f"http://{ip_port}/api/generate"

In [10]:
import json
import pandas as pd
import requests

def extract_full_text(response_text):
    """
    This function extracts and concatenates text from the 'response' field in each JSON line of the response.
    
    Parameters:
    response_text (str): The raw response text, with each line being a JSON string.

    Returns:
    str: The concatenated text from all 'response' fields.
    """
    full_text = ""  # Variable to store the complete text
    
    # Iterate over each line in the response
    for line in response_text.text.splitlines():
        try:
            data = json.loads(line)  # Parse each JSON line
            if "response" in data:  # Check if the "response" field exists
                full_text += data["response"]  # Append the text
        except json.JSONDecodeError:
            print("Unable to parse line:", line)
    
    return full_text

In [11]:

def check_relevance(abstract,title,inclusion_criteria,exclusion_criteria):
    prompt = (
        "You are a researcher rigorously screening titles and abstracts of scientific papers for inclusion or exclusion in a review paper. "
        "Use the criteria below to inform your decision. If any exclusion criteria are met or not all inclusion criteria are met, exclude the article. "
        "If all inclusion criteria are met, include the article. Only type 'included' or 'excluded' to indicate your decision. Do not type anything else.\n\n"
        f"Title: {title}\n"
        f"Abstract: {abstract}\n\n"
        f"Inclusion criteria: {inclusion_criteria}\n"
        f"Exclusion criteria: {exclusion_criteria}\n\n"
        "Decision: {here}"
        "Explain: {here}"
    )
    
    payload = {
        "model": "phi4",
        "prompt": prompt,
        "temperature": 0.2,  
        "top_p": 0.2       
    }

    # Make a POST request
    response = requests.post(url, json=payload)

    # Validate the response
    if response.status_code == 200:
        # decision = response.json().get("choices", [{}])[0].get("text", "").strip()
        decision = extract_full_text(response)
        # print(decision)
        # print("------------------------------------------------")
        return decision #if decision in {"included", "excluded"} else decision
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

In [12]:
inclusion_criteria = (
    "1. The article discusses barriers or facilitators to nurses' involvement in research activities.\n"
    "2. The article identifies challenges nurses face in conducting research, such as educational deficiencies, time constraints, organizational or cultural barriers, or lack of resources.\n"
    "3. The article proposes solutions or strategies to overcome barriers to nursing research participation.\n"
    "4. The article addresses the integration of research findings into clinical practice (Evidence-Based Practice).\n"
    "5. The article examines organizational cultures, policies, or leadership styles affecting nurses’ research involvement.\n"

)

#    "6. The article explores the use of technology, such as NLP tools, to enhance nursing research or systematic reviews."

exclusion_criteria = (

    "1. The article does not focus on nursing research or its associated barriers and solutions.\n"
    "2. The article lacks sufficient detail to evaluate against the inclusion criteria."
)

#    "1. The article is not written in English.\n"
    # "1. The article is a review paper or meta-analysis.\n"


In [14]:
check_relevance(df.iloc[0].Abstract, df.iloc[0].Title, inclusion_criteria, exclusion_criteria)

'**Decision:** Excluded\n\n**Explanation:** \n\nThe abstract of the study titled "Missed nursing care: a qualitative study" focuses on identifying elements of regularly missed nursing care and reasons for missing such care as reported by nursing staff. It does not discuss barriers or facilitators to nurses\' involvement in research activities, challenges nurses face specifically in conducting research, propose solutions to overcome these barriers, address the integration of research findings into clinical practice, nor examine organizational cultures, policies, or leadership styles affecting nurses’ research involvement.\n\nThe study\'s primary focus is on missed nursing care rather than nursing research participation or its associated factors. Therefore, it does not meet any of the inclusion criteria provided for the review paper concerning nursing research barriers and facilitators. As a result, this article should be excluded based on the specified exclusion criterion that states if

In [32]:
df = pd.read_pickle('combined_df_all_databases_only_english.pkl')


In [33]:
df.shape

(15958, 8)

In [ ]:
from tqdm import tqdm

# Define a function to process each row and update the CSV incrementally
def add_decision_column_and_save(df, inclusion_criteria, exclusion_criteria, output_file):
    # Check if the file already exists
    import os
    if os.path.exists(output_file):
        # If the file exists, load it
        processed_df = pd.read_csv(output_file)
        start_index = len(processed_df)  # Start from where we left off
    else:
        # If the file doesn't exist, initialize an empty DataFrame
        processed_df = pd.DataFrame(columns=df.columns.tolist() + ['Decision'])
        start_index = 0

    # Iterate over rows with tqdm for progress tracking, starting from the last processed row
    for i in tqdm(range(start_index, len(df)), desc="Processing rows"):
        # Extract abstract and title for each row
        abstract = df.iloc[i].Abstract
        title = df.iloc[i].Title
        
        # Call the check_relevance function
        decision = check_relevance(abstract, title, inclusion_criteria, exclusion_criteria)
        
        # Add the current row and decision to the processed DataFrame
        current_row = df.iloc[i].to_dict()
        current_row['Decision'] = decision
        processed_df = pd.concat([processed_df, pd.DataFrame([current_row])], ignore_index=True)
        
        # Save the updated DataFrame to the CSV file after each iteration
        processed_df.to_csv(output_file, index=False,encoding='utf-8')

    return processed_df


# File to save the updated DataFrame
output_file = "updated_decisions_incremental.csv"
output_file = "filter/updated_decisions_incremental_fix_2.csv"

# Call the function with your DataFrame
df_with_decisions = add_decision_column_and_save(df, inclusion_criteria, exclusion_criteria, output_file)

# Display the first few rows of the final DataFrame
print(df_with_decisions.head())


In [21]:

df1 = pd.read_csv("filter/updated_decisions_incremental_fix_2.csv")
df2 = pd.read_csv("filter/updated_decisions_incremental_fix_3.csv")
df3 = pd.read_csv("filter/updated_decisions_incremental_fix_4.csv")

df_part1 = df1.iloc[0:9000]     
df_part2 = df2    
df_part3 = df3.iloc[0:1000]          

combined_df = pd.concat([df_part1, df_part2, df_part3], ignore_index=True)

In [22]:
df_with_decisions = combined_df

In [23]:
import re

# Define a function to extract "Inclusion" or "Exclusion" from the Decision column
def extract_decision(decision_text):
    decision_text = str(decision_text)
    match = re.search(r'\b(Inclusion|Exclusion|Included|Excluded)\b', decision_text, re.IGNORECASE)
    if match:
        return match.group(0).capitalize()  # Capitalize to maintain consistency (e.g., "Excluded")
    return "Unknown"  # Return "Unknown" if no match is found

# Apply the function to the Decision column
df_with_decisions['Decision Type'] = df_with_decisions['Decision'].apply(extract_decision)

# Display the DataFrame with the extracted decision type
print(df_with_decisions[['Decision', 'Decision Type']].head())


                                            Decision Decision Type
0  Decision: excluded\n\nExplain: The title and a...      Excluded
1  **Decision:** Excluded\n\n**Explanation:**  \n...      Excluded
2  **Decision:** Excluded\n\n**Explanation:** \n\...      Excluded
3  **Decision:** Excluded\n\n**Explanation:**  \n...      Excluded
4  **Decision:** Excluded\n\n**Explanation:**  \n...      Excluded


In [36]:
# df_with_decisions['Decision Type'].value_counts()


In [33]:
# df_with_decisions[df_with_decisions['Decision Type'] == 'Excluded'].sample(100).to_csv('100_Excluded.csv')

In [39]:
# df_with_decisions[(df_with_decisions['Decision Type'] == 'Included') | (df_with_decisions['Decision Type'] == 'Inclusion')].to_csv('filter/Labeled_539_Included_1.csv',encoding='utf-8')

In [41]:
import pandas as pd
import unicodedata
import re

# Define a function to normalize titles (Unicode normalization, remove punctuation, collapse whitespace)
def normalize_title(text):
    if pd.isna(text):
        return ''
    text = unicodedata.normalize('NFKC', text)      # Normalize all Unicode variants
    text = text.lower().strip()                     # Lowercase and remove leading/trailing spaces
    text = re.sub(r'\s+', ' ', text)                # Convert multiple whitespace (including thin space) to regular space
    text = re.sub(r'[^\w\s]', '', text)             # Remove punctuation
    return text

# Step 1: Apply normalization to the Title column
included_df['Title_cleaned'] = included_df['Title'].apply(normalize_title)

# Step 2: Remove duplicates based on the cleaned title
included_df_unique_by_title = included_df.drop_duplicates(subset='Title_cleaned', keep='first')

# Step 3: Drop the helper column before saving (optional)
included_df_unique_by_title = included_df_unique_by_title.drop(columns=['Title_cleaned'])

# Step 4: Save to Excel
included_df_unique_by_title.to_excel('filter/Labeled_535_Included_1.xlsx', index=False)

print(f"✅ Saved {included_df_unique_by_title.shape[0]} unique rows (based on cleaned title) to Excel.")


/tmp/ipykernel_3825556/2748352577.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  included_df['Title_cleaned'] = included_df['Title'].apply(normalize_title)


✅ Saved 535 unique rows (based on cleaned title) to Excel.
